In [1]:
import gradio as gr
from openai import OpenAI
import gradio as gr
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [2]:
model_1b ="llama3.2:1b"

base_url = "http://localhost:11434/v1"

ollama = OpenAI(base_url=base_url, api_key="ollama")

In [4]:
system_message = "Koristi sljedeće dijelove konteksta da odgovoriš na korisnikovo pitanje. "

system_message += "Ako ne znaš odgovor, samo odgovori da ne znaš, nemoj pokušavati izmišljati odgovore."

print(system_message)

Koristi sljedeće dijelove konteksta da odgovoriš na korisnikovo pitanje. Ako ne znaš odgovor, samo odgovori da ne znaš, nemoj pokušavati izmišljati odgovore.


In [3]:
embeding_model = "jeffh/intfloat-multilingual-e5-large-instruct:f32"

db_name = "vector_db"

In [5]:
embeddings = OllamaEmbeddings(model=embeding_model)

In [6]:
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)
print(f"Vectorstore found with {vectorstore._collection.count()} documents")

retriever = vectorstore.as_retriever() # dodati kwargs

Vectorstore found with 323 documents


In [7]:
retrieved_documents = retriever.invoke("Šta je refrentni model?")

In [9]:
retrieved_documents[1]

Document(id='1f8868a4-03f0-475f-a1f7-6d907c72659c', metadata={'producer': 'Microsoft® Office Word 2007', 'title': 'Naslov (Verdana 11 bold)', 'author': 'Jasmin', 'source': 'my-knowledge\\računarske_mreže\\Transportni_sloj.pdf', 'creator': 'Microsoft® Office Word 2007', 'total_pages': 7, 'creationdate': '2009-12-27T23:40:18+00:00', 'name': 'Transportni_sloj', 'moddate': '2015-05-24T22:19:41+02:00', 'category': 'računarske_mreže'}, page_content='Računarske mreže::Workshop \n\nCopyright © by: FIT  \n\n7')

In [10]:
def make_context(similars):
    message = "\nIspod se nalazi kontekst koji će ti pomoći da odgovoriš na pitanje.\n\n"
    for similar in similars:
        message += similar.page_content
    return message

In [11]:
print(make_context(retrieved_documents))


Ispod se nalazi kontekst koji će ti pomoći da odgovoriš na pitanje.

3.2 Referentni modeli 

U naredna dva  odjeljka biće  obrađene dvije  važne arhitekture  mreže,  odnosno  referentni 
modeli OSI i TCP/IP. Iako se protokoli povezani s modelom OSI danas rijetko koriste, sam 
model je sveobuhvatan i još uvijek važeći, a svojstva svakog sloja i dalje su veoma važna. 
TCP/IP se na neki način nalazi na suprotnom kraju: sam model nema širu primjenu ali se 
njegovi protokoli nalaze svuda.  

3.2.1 OSI referentni model 

Na donjoj slici prikazan je OSI model (bez fizičkog medijuma). Model se zasniva na prijed-
logu Međunarodne organizacije za standardizaciju (ISO) i trebalo je da bude prvi korak ka 
međunarodnom standardiziranju protokola koji se koriste u različitim slojevima. Prepravljen 
je  1995.  godine.  Model  se  zove  ISO  OSI  (International  Standards  Organization  –  Open 
Systems Interconnection), jer treba povezati otvorene sisteme (otvoreni za komuniciranje 
s drugim sistemi

In [12]:
def chat(message, history):
    history = [{'role' : h['role'], 'content': h['content']} for h in history]
    similar_documents = retriever.invoke(message)
    message += make_context(similar_documents) 
    messages = [{'role' : 'system', 'content': system_message}] + history + [{'role' : 'user', 'content': message}]
    stream = ollama.chat.completions.create(model=model_1b, messages=messages, stream=True)
    response = ''
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
def chat(message, history):
    history = [{'role':h['role'], 'content': h['content']} for h in history]
    messages = [{'role':'system', 'content': system_message}] + history + [{'role':'user', 'content': message}]
    stream = ollama.chat.completions.create(model = model_1b, messages=messages, stream=True)
    response = ''
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat).launch(inbrowser=True)